## Previsão Multi-Step — 3 meses à frente

**Correções aplicadas:**
1. Removido `Target = shift(-1)`: o modelo agora prevê `Qtde` diretamente, como todos os outros notebooks.
   A abordagem com `shift(-1)` gerava um target diferente e tornava os modelos incomparáveis entre si.
2. `Media_3` com `shift(1)` antes do rolling — sem data leakage.
3. Lógica recursiva de lags corrigida para ser consistente com o target.


In [1]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

engine = create_engine("sqlite:///../data/DBVendas.db")


In [2]:
query = """
SELECT
    pd.Data_Venda,
    iv.ID_Produto,
    iv.Qtde
FROM itens_vendas iv
JOIN vendas pd ON iv.ID_Pedido = pd.ID_Pedido
"""

df = pd.read_sql(query, engine)
df['Data_Venda'] = pd.to_datetime(df['Data_Venda'])
df['Ano'] = df['Data_Venda'].dt.year
df['Mes'] = df['Data_Venda'].dt.month

print(f"✅ Dados: {df.shape}")


✅ Dados: (100000, 5)


In [3]:
# ==============================
# AGREGAÇÃO MENSAL
# ==============================
df_agg = df.groupby(['ID_Produto', 'Ano', 'Mes'])['Qtde'].sum().reset_index()

df_agg['Data'] = pd.to_datetime(
    df_agg['Ano'].astype(str) + '-' + df_agg['Mes'].astype(str) + '-01'
)

df_agg = df_agg.sort_values(['ID_Produto', 'Data'])


In [4]:
# ==============================
# FEATURES (SEM LEAKAGE)
# ==============================
df_agg['Lag_1'] = df_agg.groupby('ID_Produto')['Qtde'].shift(1)
df_agg['Lag_2'] = df_agg.groupby('ID_Produto')['Qtde'].shift(2)

# ✅ CORRIGIDO: shift(1) antes do rolling
df_agg['Media_3'] = (
    df_agg.groupby('ID_Produto')['Qtde']
    .transform(lambda x: x.shift(1).rolling(3).mean())
)

df_agg = df_agg.dropna()

print(f"✅ Dataset pronto: {df_agg.shape}")
print(df_agg.head(6))


✅ Dataset pronto: (28751, 8)
   ID_Produto   Ano  Mes  Qtde       Data  Lag_1  Lag_2    Media_3
3        1010  2010    4    44 2010-04-01   31.0   44.0  38.666667
4        1010  2010    5    75 2010-05-01   44.0   31.0  39.666667
5        1010  2010    6    52 2010-06-01   75.0   44.0  50.000000
6        1010  2010    7    90 2010-07-01   52.0   75.0  57.000000
7        1010  2010    8    19 2010-08-01   90.0   52.0  72.333333
8        1010  2010    9    12 2010-09-01   19.0   90.0  53.666667


In [5]:
# ==============================
# SPLIT TEMPORAL
# ==============================
# ✅ CORRIGIDO: target é Qtde (direto), não shift(-1).
# shift(-1) criava um target diferente de todos os outros notebooks,
# tornando os MAEs incomparáveis e a lógica recursiva inconsistente.

cutoff = '2021-01-01'

train = df_agg[df_agg['Data'] < cutoff]
test  = df_agg[df_agg['Data'] >= cutoff]

features = ['ID_Produto', 'Ano', 'Mes', 'Lag_1', 'Lag_2', 'Media_3']

X_train = train[features]
y_train = train['Qtde']

X_test = test[features]
y_test = test['Qtde']

print(f"📊 Treino: {X_train.shape}  |  Teste: {X_test.shape}")


📊 Treino: (26310, 6)  |  Teste: (2441, 6)


In [6]:
# ==============================
# TREINAR E AVALIAR
# ==============================
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

mae      = mean_absolute_error(y_test, model.predict(X_test))
erro_pct = (mae / y_test.mean()) * 100
print(f"📉 MAE: {mae:.2f}  |  Erro %: {erro_pct:.2f}%")


📉 MAE: 21.91  |  Erro %: 49.67%


In [7]:
# ==============================
# PREVISÃO RECURSIVA — 3 MESES
# ==============================
print("🔮 Prevendo próximos 3 meses...")

horizonte = 3
previsoes = []

for produto in df_agg['ID_Produto'].unique():

    df_prod = df_agg[df_agg['ID_Produto'] == produto].sort_values('Data')

    # ✅ lag1/lag2 sempre referem à Qtde real/prevista — coerente com o target
    lag1 = df_prod.iloc[-1]['Qtde']
    lag2 = df_prod.iloc[-2]['Qtde']

    ultima = df_prod.iloc[-1]

    for step in range(1, horizonte + 1):

        mes = int(ultima['Mes']) + step
        ano = int(ultima['Ano'])

        if mes > 12:
            mes -= 12
            ano += 1

        media_3 = (lag1 + lag2 + df_prod.iloc[-3]['Qtde']) / 3

        X_pred = pd.DataFrame([{
            'ID_Produto': produto,
            'Ano'       : ano,
            'Mes'       : mes,
            'Lag_1'     : lag1,
            'Lag_2'     : lag2,
            'Media_3'   : media_3
        }])

        pred = model.predict(X_pred)[0]

        previsoes.append({
            'ID_Produto': produto,
            'Ano'       : ano,
            'Mes'       : mes,
            'Previsao'  : pred
        })

        # Atualização recursiva correta
        lag2 = lag1
        lag1 = pred

print("✅ Previsão concluída")


🔮 Prevendo próximos 3 meses...
✅ Previsão concluída


In [9]:
# ==============================
# EXPORTAR
# ==============================
df_prev = pd.DataFrame(previsoes)

df_prev.to_excel("../planilhas/Previsao_3_Meses.xlsx", index=False)
df_prev.to_sql("previsao_demanda", engine, if_exists="replace", index=False)

print("✅ Previsao_3_Meses.xlsx salvo")
print("✅ Tabela previsao_demanda salva")
print(df_prev.head(12))


✅ Previsao_3_Meses.xlsx salvo
✅ Tabela previsao_demanda salva
    ID_Produto   Ano  Mes  Previsao
0         1010  2022    1     47.62
1         1010  2022    2     55.46
2         1010  2022    3     37.07
3         1011  2022    1     54.20
4         1011  2022    2     55.49
5         1011  2022    3     44.10
6         1012  2022    1     49.06
7         1012  2022    2     44.94
8         1012  2022    3     42.10
9         1013  2022    1     40.11
10        1013  2022    2     47.30
11        1013  2022    3     53.73


In [10]:
# ==============================
# PIVOT HORIZONTAL + FORMATAÇÃO
# ==============================
df_prev["Ano"] = df_prev["Ano"].astype(int)
df_prev["Mes"] = df_prev["Mes"].astype(int)
df_prev["Periodo"] = df_prev["Mes"].apply(lambda m: f"{m:02d}") + "/" + df_prev["Ano"].astype(str)

pv = df_prev.pivot_table(
    index="ID_Produto", columns="Periodo", values="Previsao", aggfunc="sum"
).reset_index()

period_cols = sorted(
    [c for c in pv.columns if c != "ID_Produto"],
    key=lambda p: (p[3:], p[:2])
)

result = pv[["ID_Produto"] + period_cols].rename(columns={"ID_Produto": "Produto"})

output_path = "../planilhas/pivot_previsao.xlsx"
result.to_excel(output_path, index=False, sheet_name="Pivot")

# Formatação
wb = load_workbook(output_path)
ws = wb["Pivot"]

header_fill = PatternFill("solid", start_color="1F4E79")
header_font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
body_font   = Font(name="Arial", size=10)
center      = Alignment(horizontal="center", vertical="center")
right       = Alignment(horizontal="right",  vertical="center")
thin_border = Border(
    left  =Side(style="thin", color="D9D9D9"),
    right =Side(style="thin", color="D9D9D9"),
    bottom=Side(style="thin", color="D9D9D9"),
)

for cell in ws[1]:
    cell.fill      = header_fill
    cell.font      = header_font
    cell.alignment = center
    cell.border    = thin_border

for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    for col_idx, cell in enumerate(row, start=1):
        cell.font      = body_font
        cell.border    = thin_border
        cell.alignment = center if col_idx == 1 else right
        if col_idx > 1:
            cell.number_format = '#,##0.00'

ws.column_dimensions["A"].width = 14
for col_idx in range(2, ws.max_column + 1):
    ws.column_dimensions[get_column_letter(col_idx)].width = 12

ws.freeze_panes = "B2"
wb.save(output_path)

result.to_sql("previsao_pivot", engine, if_exists="replace", index=False)

print(f"✅ pivot_previsao.xlsx salvo | {len(result):,} produtos | {len(period_cols)} períodos")


✅ pivot_previsao.xlsx salvo | 212 produtos | 5 períodos
